# LangChain Travel Agent with Mem0 Memory

A simple travel-agent chatbot built with LangChain that remembers user preferences across turns (and across sessions) using Mem0. Based on the [Mem0 LangChain guide](https://docs.mem0.ai/integrations/langchain).

In [ ]:
%pip install -qU langchain langchain-openai mem0ai python-dotenv

## API keys

Reads `OPENAI_API_KEY` and `MEM0_API_KEY` from `.env` / the environment, and prompts only if missing.

In [9]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

for key in ("OPENAI_API_KEY", "MEM0_API_KEY"):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"{key}: ")

In [10]:
from typing import List, Dict

from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from mem0 import MemoryClient

## Clients and prompt

The prompt has a placeholder where retrieved memories are injected as context.

In [ ]:
llm = ChatOpenAI(model="gpt-5-nano")
mem0 = MemoryClient()

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=(
        "You are a helpful travel agent AI. Use the provided context to personalize "
        "your responses and remember user preferences and past interactions. "
        "Provide travel recommendations, itinerary suggestions, and answer questions "
        "about destinations. Keep the response crisp and clean. No bloated information"
    )),
    MessagesPlaceholder(variable_name="context"),
    ("human", "{input}"),
])

## Memory helpers

- `retrieve_context` searches Mem0 for memories relevant to the user's message.
- `save_interaction` stores the user/assistant exchange so Mem0 can extract new memories.

In [12]:
def retrieve_context(query: str, user_id: str) -> List[Dict]:
    """Search Mem0 for memories relevant to the query."""
    try:
        memories = mem0.search(query, filters={"user_id": user_id})
        serialized = " ".join(m["memory"] for m in memories["results"])
        return [{"role": "system", "content": f"Relevant information: {serialized}"}]
    except Exception as e:
        print(f"Error retrieving memories: {e}")
        return []


def save_interaction(user_id: str, user_input: str, assistant_response: str):
    try:
        mem0.add(
            [
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": assistant_response},
            ],
            user_id=user_id,
        )
    except Exception as e:
        print(f"Error saving interaction: {e}")

## One chat turn

Retrieve memories → generate a reply → save the exchange.

In [13]:
def generate_response(user_input: str, context: List[Dict]) -> str:
    chain = prompt | llm
    return chain.invoke({"context": context, "input": user_input}).content


def chat_turn(user_input: str, user_id: str) -> str:
    context = retrieve_context(user_input, user_id)
    response = generate_response(user_input, context)
    save_interaction(user_id, user_input, response)
    return response

## Interactive chat

Run this cell to chat; type `quit` to stop.

In [15]:
user_id = "shiva"

while True:
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit", "bye"):
        print("Travel Agent: Thank you for using our service. Have a great trip!")
        break
    print(f"You: {user_input}")
    print(f"Travel Agent: {chat_turn(user_input, user_id)}")

Travel Agent: Hi Shiva! Great idea for a quiet, beachy winter escape. Here are three calm, secluded beach destinations you could consider this winter. Each one is known for its peaceful vibe and relatively uncrowded shores. If you’d prefer, I can tailor these into a single 2–3 week itinerary with smart routing and lodging options.

Option 1: Havelock Island, Andaman & Nicobar Islands, India
- Why it fits: Pristine white-sand beaches, turquoise water, and a slow pace. Radhanagar Beach is iconic but there are quieter coves like Vijaynagar and Kalapathar Beach. Excellent snorkeling near Elephant Beach.
- Best time: November–February (cooler, dry, calm seas).
- Getting there: Fly to Port Blair (Verious Indian cities have direct or connecting flights). From Port Blair, a 2–3 hour ferry ride to Havelock.
- What to do: Beach days, snorkeling, kayaking, relaxed seaside meals, sunset views.
- Where to stay: Eco-friendly beachfront cottages and small boutique resorts.
- Suggested length: 4–6 day